In [0]:
dbutils.widgets.text("run_date","")

run_date = dbutils.widgets.get("run_date")


print(f"run_date: {run_date}")

In [0]:
# Service Principal credentials
application_id = "********************************"
authentication_key = "***************************"
tenant_id = "*********************************"

# Set Spark config for ADLS access
spark.conf.set("fs.azure.account.auth.type.petroflowstorage.dfs.core.windows.net", "OAuth")
spark.conf.set("fs.azure.account.oauth.provider.type.petroflowstorage.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set("fs.azure.account.oauth2.client.id.petroflowstorage.dfs.core.windows.net", application_id)
spark.conf.set("fs.azure.account.oauth2.client.secret.petroflowstorage.dfs.core.windows.net", authentication_key)
spark.conf.set("fs.azure.account.oauth2.client.endpoint.petroflowstorage.dfs.core.windows.net", "https://login.microsoftonline.com/" + tenant_id + "/oauth2/token")

In [0]:
oil_silver = "abfss://silver@petroflowstorage.dfs.core.windows.net/oil-prices/"
gas_silver ="abfss://silver@petroflowstorage.dfs.core.windows.net/natural-gas/"

df_oil = spark.read.parquet(oil_silver)
df_gas = spark.read.parquet(gas_silver)

print(f"oil_silver: {df_oil.count()}")
print(f"gas_silver: {df_gas.count()}")

oil_silver: 5000
gas_silver: 4996


In [0]:
df_combined=df_oil.union(df_gas)

print(f"df_combined: {df_combined.count()}")

df_combined.display()

df_combined: 9996


trade_date,price_usd,series,series_description,units,area,product_name,energy_type,source_system,ingestion_date
2025-11-04,2.54,EER_EPD2DC_PF4_Y05LA_DPG,"Los Angeles, CA Ultra-Low Sulfur CARB Diesel Spot Price (Dollars per Gallon)",$/GAL,Y05LA,Carb Diesel,CRUDE_OIL,EIA_API,2026-06-04
2025-11-05,2.63,EER_EPD2DC_PF4_Y05LA_DPG,"Los Angeles, CA Ultra-Low Sulfur CARB Diesel Spot Price (Dollars per Gallon)",$/GAL,Y05LA,Carb Diesel,CRUDE_OIL,EIA_API,2026-06-04
2025-11-06,2.7,EER_EPD2DC_PF4_Y05LA_DPG,"Los Angeles, CA Ultra-Low Sulfur CARB Diesel Spot Price (Dollars per Gallon)",$/GAL,Y05LA,Carb Diesel,CRUDE_OIL,EIA_API,2026-06-04
2025-11-07,2.68,EER_EPD2DC_PF4_Y05LA_DPG,"Los Angeles, CA Ultra-Low Sulfur CARB Diesel Spot Price (Dollars per Gallon)",$/GAL,Y05LA,Carb Diesel,CRUDE_OIL,EIA_API,2026-06-04
2025-11-10,2.7,EER_EPD2DC_PF4_Y05LA_DPG,"Los Angeles, CA Ultra-Low Sulfur CARB Diesel Spot Price (Dollars per Gallon)",$/GAL,Y05LA,Carb Diesel,CRUDE_OIL,EIA_API,2026-06-04
2025-11-12,2.67,EER_EPD2DC_PF4_Y05LA_DPG,"Los Angeles, CA Ultra-Low Sulfur CARB Diesel Spot Price (Dollars per Gallon)",$/GAL,Y05LA,Carb Diesel,CRUDE_OIL,EIA_API,2026-06-04
2024-09-04,2.22,EER_EPD2DC_PF4_Y05LA_DPG,"Los Angeles, CA Ultra-Low Sulfur CARB Diesel Spot Price (Dollars per Gallon)",$/GAL,Y05LA,Carb Diesel,CRUDE_OIL,EIA_API,2026-06-04
2024-09-05,2.23,EER_EPD2DC_PF4_Y05LA_DPG,"Los Angeles, CA Ultra-Low Sulfur CARB Diesel Spot Price (Dollars per Gallon)",$/GAL,Y05LA,Carb Diesel,CRUDE_OIL,EIA_API,2026-06-04
2024-09-06,2.19,EER_EPD2DC_PF4_Y05LA_DPG,"Los Angeles, CA Ultra-Low Sulfur CARB Diesel Spot Price (Dollars per Gallon)",$/GAL,Y05LA,Carb Diesel,CRUDE_OIL,EIA_API,2026-06-04
2024-09-09,2.21,EER_EPD2DC_PF4_Y05LA_DPG,"Los Angeles, CA Ultra-Low Sulfur CARB Diesel Spot Price (Dollars per Gallon)",$/GAL,Y05LA,Carb Diesel,CRUDE_OIL,EIA_API,2026-06-04


**MONTHLY PRICE TRENDS**

In [0]:
from pyspark.sql.functions import date_format,avg,max,min,count,round

df_monthly = df_combined.\
    withColumn("year_month", date_format("trade_date", "yyyy-MM")).\
    groupBy("year_month", "energy_type").\
    agg(
        round(avg("price_usd"), 2).alias("avg_price_usd"),
        round(max("price_usd"), 2).alias("max_price_usd"),
        round(min("price_usd"), 2).alias("min_price_usd"),
        count("*").alias("trading_days")
    ).\
    orderBy("year_month", "energy_type")

display(df_monthly)

print(f"monthly trend records:{df_monthly.count()}")

year_month,energy_type,avg_price_usd,max_price_usd,min_price_usd,trading_days
1989-01,NATURAL_GAS,6.57,14.67,3.53,12
1989-02,NATURAL_GAS,6.56,14.31,3.51,12
1989-03,NATURAL_GAS,6.63,15.12,3.56,12
1989-04,NATURAL_GAS,6.91,15.49,3.61,12
1989-05,NATURAL_GAS,7.44,15.75,3.72,12
1989-06,NATURAL_GAS,7.84,16.23,3.84,12
1989-07,NATURAL_GAS,8.21,16.37,4.07,12
1989-08,NATURAL_GAS,8.31,16.47,4.19,12
1989-09,NATURAL_GAS,8.27,15.98,3.94,12
1989-10,NATURAL_GAS,7.92,15.99,3.71,12


monthly trend records:693


**PRICE CHANGE ANALYSIS**


In [0]:
from pyspark.sql.window import Window

from pyspark.sql.functions import lag,round,col

window = Window.partitionBy("energy_type").orderBy("year_month")

df_price_change= df_monthly\
    .withColumn("prev_month_price",lag("avg_price_usd",1).over(window))\
    .withColumn("price_change",
               round((col("avg_price_usd")-col("prev_month_price"))/col("prev_month_price"),2))\
                   .withColumn("price_change_pct", round(col("price_change")/col("prev_month_price")*100,2))\
                       .filter(col("prev_month_price").isNotNull())


df_price_change.display()


year_month,energy_type,avg_price_usd,max_price_usd,min_price_usd,trading_days,prev_month_price,price_change,price_change_pct
1996-05,CRUDE_OIL,0.87,0.98,0.79,22,0.95,-0.08,-8.42
1996-06,CRUDE_OIL,0.75,0.85,0.66,20,0.87,-0.14,-16.09
1996-07,CRUDE_OIL,0.69,0.74,0.64,22,0.75,-0.08,-10.67
1996-08,CRUDE_OIL,0.71,0.73,0.69,21,0.69,0.03,4.35
1996-09,CRUDE_OIL,0.77,0.79,0.74,20,0.71,0.08,11.27
1996-10,CRUDE_OIL,0.82,0.85,0.78,23,0.77,0.06,7.79
1996-11,CRUDE_OIL,0.74,0.79,0.71,20,0.82,-0.1,-12.2
1996-12,CRUDE_OIL,0.74,0.75,0.71,21,0.74,0.0,0.0
1997-01,CRUDE_OIL,0.77,0.89,0.73,22,0.74,0.04,5.41
1997-02,CRUDE_OIL,0.79,0.86,0.72,19,0.77,0.03,3.9


**YEAR SUMMARY KPIS**

In [0]:
from pyspark.sql.functions import year

df_yearly = df_combined.\
    withColumn("year", year("trade_date")).\
    groupBy("year", "energy_type").\
    agg(
        round(avg("price_usd"), 2).alias("avg_price_usd"),
        round(max("price_usd"), 2).alias("max_price_usd"),
        round(min("price_usd"), 2).alias("min_price_usd"),
        count("*").alias("total_records")
    ).\
    orderBy("year", "energy_type")

display(df_yearly)

year,energy_type,avg_price_usd,max_price_usd,min_price_usd,total_records
1989,NATURAL_GAS,7.37,16.47,3.51,144
1990,NATURAL_GAS,7.55,20.13,3.68,144
1991,NATURAL_GAS,8.23,24.82,3.7,144
1992,NATURAL_GAS,7.87,18.6,3.68,144
1993,NATURAL_GAS,7.98,18.43,3.82,140
1994,NATURAL_GAS,7.73,12.86,3.48,132
1995,NATURAL_GAS,7.4,12.22,3.51,132
1996,CRUDE_OIL,0.77,0.98,0.64,179
1996,NATURAL_GAS,7.82,13.65,3.3,132
1997,CRUDE_OIL,0.68,0.89,0.55,252


**TO GOLD LAYER**


In [0]:
df_monthly.write\
    .mode("overwrite")\
    .parquet("abfss://gold@petroflowstorage.dfs.core.windows.net/energy_kpis/monthly-trends/")
df_yearly.write\
    .mode("overwrite")\
        .parquet("abfss://gold@petroflowstorage.dfs.core.windows.net/energy_kpis/yearly-trends/")
df_price_change.write\
    .mode("overwrite")\
        .parquet("abfss://gold@petroflowstorage.dfs.core.windows.net/energy_kpis/price-change/")

print("Data written to gold layer")

Data written to gold layer


In [0]:
df_monthly_verify = spark.read.parquet("abfss://gold@petroflowstorage.dfs.core.windows.net/energy_kpis/monthly-trends/")
df_yearly_verify = spark.read.parquet("abfss://gold@petroflowstorage.dfs.core.windows.net/energy_kpis/yearly-trends/")
df_price_change_verify = spark.read.parquet("abfss://gold@petroflowstorage.dfs.core.windows.net/energy_kpis/price-change/")

print(f"df_monthly_verify: {df_monthly_verify.count()}")
print(f"df_yearly_verify: {df_yearly_verify.count()}")
print(f"df_price_change_verify: {df_price_change_verify.count()}")

df_monthly_verify: 693
df_yearly_verify: 61
df_price_change_verify: 691
